# Практика 6. Основные структуры pandas: Series и DataFrame

На практике 2 мы считали отчёт вручную: цикл, условие, словарь-накопитель. Сегодня тот же набор
операций мы делаем инструментом, который для этого создан, — и на порядок быстрее.

**Данные пары:** журнал грузовых рейсов транспортной компании за март 2025 — 400 рейсов,
маршруты, перевозчики, вес, расстояние, стоимость, задержки.

**Из чего состоит пара:**

| Часть | О чём |
|---|---|
| 1 | `Series` — одномерная структура: создание и исследование |
| 2 | `DataFrame` — таблица: создание и первичное исследование |
| 3 | Индексация: `loc` и `iloc` |
| 4 | Фильтрация и выбор данных |
| 5 | Изменение структуры и вычисляемые признаки |
| 6 | Применение функций к строкам и столбцам |
| 7 | Мини-проект: карточка маршрута |

> Ячейки запускаются **Shift + Enter**, строго по порядку: каждая следующая использует
> переменные из предыдущих.

---
## Шаг 0. Проект (5 минут, повторение)

```
# терминал
mkdir pandas-trips
cd pandas-trips
```

`requirements.txt`:

```
pandas
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

`.gitignore` с `.venv/`, `git init`, ноутбук в папку проекта, kernel — `.venv`.

---
# Часть 1. Series

**`Series`** — одномерный набор значений **с индексом**. Это не просто список: у каждого значения
есть метка, по которой его можно достать, и все значения одного типа.

Проще всего думать так: `Series` — это один столбец таблицы, а `DataFrame` — таблица целиком.

Главное свойство, из-за которого pandas вообще нужен: **операции применяются сразу ко всем
элементам**, без цикла. Это называется векторизацией.

In [2]:
import pandas as pd

# Из списка: индекс создастся автоматически — 0, 1, 2, ...
weights = pd.Series([12.5, 8.0, 21.3, 15.1])
print(weights)
print()

# Из словаря: ключи становятся индексом
tariffs = pd.Series({
    "Контейнер": 7.5,
    "Нефтепродукты": 9.2,
    "Зерно": 5.1,
    "Металл": 6.8,
})
print(tariffs)
print()

print("индекс:  ", list(tariffs.index))
print("значения:", list(tariffs.values))
print("тип значений:", tariffs.dtype)
print("размер:", tariffs.size)

0    12.5
1     8.0
2    21.3
3    15.1
dtype: float64

Контейнер        7.5
Нефтепродукты    9.2
Зерно            5.1
Металл           6.8
dtype: float64

индекс:   ['Контейнер', 'Нефтепродукты', 'Зерно', 'Металл']
значения: [np.float64(7.5), np.float64(9.2), np.float64(5.1), np.float64(6.8)]
тип значений: float64
размер: 4


In [6]:
# Доступ: по метке и по позиции
print("по метке 'Зерно':      ", tariffs["Зерно"])
print("по позиции (iloc[2]):  ", tariffs.iloc[2])
print()

# Векторизация: операция применяется ко всем элементам сразу, цикл не нужен
print("тарифы +10%:")
print((tariffs * 1.1).round(2))
print()

# Встроенная статистика
print("среднее:", tariffs.mean().round(2))
print("максимум:", tariffs.max(), "— это", tariffs.idxmax())
print()
print(tariffs.describe().round(2))

по метке 'Зерно':       5.1
по позиции (iloc[2]):   5.1

тарифы +10%:
Контейнер         8.25
Нефтепродукты    10.12
Зерно             5.61
Металл            7.48
dtype: float64

среднее: 7.15
максимум: 9.2 — это Нефтепродукты

count    4.00
mean     7.15
std      1.70
min      5.10
25%      6.38
50%      7.15
75%      7.92
max      9.20
dtype: float64


### Задание 1

Дан `Series` с весом груза по рейсам (индекс — номер рейса).

1. Выведите вес рейса `T003` по метке.
2. Выведите вес второго рейса по позиции.
3. Посчитайте суммарный и средний вес.
4. Создайте новый `Series` — стоимость перевозки, если тариф 6.8 руб. за тонно-километр,
   а расстояние для всех рейсов 500 км. Округлите до целых.

In [10]:
cargo = pd.Series([12.5, 8.0, 21.3, 15.1, 4.2],
                  index=["T001", "T002", "T003", "T004", "T005"])
print(cargo)

# 1) вес рейса T003 по метке
print("T003:", cargo["T003"])

# 2) вес второго рейса по позиции (помните: позиции считаются с нуля)
print("второй рейс:", cargo.iloc[1])

# 3) суммарный и средний вес
print("сумма:", cargo.sum())
print("среднее:", cargo.mean())

# 4) стоимость перевозки: вес * 500 км * 6.8 руб.
cost = cargo * 500 * 6.8
print(cost)

T001    12.5
T002     8.0
T003    21.3
T004    15.1
T005     4.2
dtype: float64
T003: 21.3
второй рейс: 8.0
сумма: 61.1
среднее: 12.22
T001    42500.0
T002    27200.0
T003    72420.0
T004    51340.0
T005    14280.0
dtype: float64


---
# Часть 2. DataFrame

**`DataFrame`** — двумерная таблица: строки с индексом и именованные столбцы.
Каждый столбец — это `Series`, и у всех столбцов общий индекс.

Создать его можно из словаря (ключ — имя столбца), из списка словарей (каждый словарь — строка)
или прочитать из файла.

In [3]:
# Способ 1: словарь списков — ключи становятся названиями столбцов
df_small = pd.DataFrame({
    "trip_id": ["T001", "T002", "T003"],
    "route": ["Москва-Казань", "Москва-Воронеж", "Казань-Самара"],
    "weight_t": [12.5, 8.0, 21.3],
})
print(df_small)
print()

# Способ 2: список словарей — каждый словарь это строка
df_small2 = pd.DataFrame([
    {"trip_id": "T001", "weight_t": 12.5},
    {"trip_id": "T002", "weight_t": 8.0},
])
print(df_small2)
print()

print("размер (строки, столбцы):", df_small.shape)
print("столбцы:", list(df_small.columns))

  trip_id           route  weight_t
0    T001   Москва-Казань      12.5
1    T002  Москва-Воронеж       8.0
2    T003   Казань-Самара      21.3

  trip_id  weight_t
0    T001      12.5
1    T002       8.0

размер (строки, столбцы): (3, 3)
столбцы: ['trip_id', 'route', 'weight_t']


## Данные пары

Запустите ячейку ниже — она создаст `data/trips.csv`.

| Колонка | Смысл |
|---|---|
| `trip_id` | номер рейса |
| `date` | дата рейса |
| `route` | маршрут |
| `carrier` | перевозчик |
| `cargo_type` | тип груза |
| `weight_t` | вес груза, тонн (местами не заполнен) |
| `distance_km` | расстояние, км |
| `duration_h` | время в пути, часов |
| `cost_rub` | стоимость перевозки, руб. |
| `delay_h` | задержка, часов (0 — рейс по расписанию) |

In [4]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips.csv — журнал грузовых рейсов транспортной компании за март 2025.

import csv
import random
from pathlib import Path

random.seed(6)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
# тариф в рублях за тонно-километр
tariffs = {
    "Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
    "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9,
}

rows = []
for i in range(1, 401):
    route, distance = random.choice(list(routes.items()))
    cargo = random.choice(list(tariffs))
    carrier = random.choice(carriers)

    weight = round(random.uniform(3, 22), 1)
    speed = random.uniform(42, 68)
    delay = 0 if random.random() < 0.7 else random.choice([1, 2, 3, 4, 6, 8, 12])
    duration = round(distance / speed + delay, 1)
    cost = round(distance * weight * tariffs[cargo] * random.uniform(0.9, 1.15), -1)

    rows.append({
        "trip_id": f"T{i:04d}",
        "date": f"2025-03-{random.randint(1, 31):02d}",
        "route": route,
        "carrier": carrier,
        "cargo_type": cargo,
        # в 3% случаев вес не записали — пригодится, когда будем смотреть пропуски
        "weight_t": "" if random.random() < 0.03 else weight,
        "distance_km": distance,
        "duration_h": duration,
        "cost_rub": int(cost),
        "delay_h": delay,
    })

Path("data").mkdir(exist_ok=True)
with open("data/trips.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips.csv")
print("Строк:", len(rows))
print("Колонки:", list(rows[0].keys()))

Готово. Создан файл data/trips.csv
Строк: 400
Колонки: ['trip_id', 'date', 'route', 'carrier', 'cargo_type', 'weight_t', 'distance_km', 'duration_h', 'cost_rub', 'delay_h']


## Первичное исследование таблицы

Четыре команды, с которых начинается работа с **любой** незнакомой таблицей:

| Команда | Что показывает |
|---|---|
| `df.shape` | сколько строк и столбцов |
| `df.head()` | первые 5 строк — глазами посмотреть на данные |
| `df.info()` | типы столбцов и количество заполненных значений |
| `df.describe()` | статистику по числовым столбцам |

Отдельно — `df["колонка"].value_counts()` для категорий и `df.isna().sum()` для пропусков.

In [5]:
df = pd.read_csv("data/trips.csv", parse_dates=["date"])

print("размер:", df.shape)
df.head()

размер: (400, 10)


,trip_id,date,route,carrier,cargo_type,weight_t,distance_km,duration_h,cost_rub,delay_h
0,T0001,2025-03-24,Москва-Санкт-Петербург,СибПуть,Металл,3.7,710,15.5,20390,0
1,T0002,2025-03-22,Москва-Казань,ВолгаТранс,Зерно,18.2,820,13.4,78740,0
2,T0003,2025-03-26,Казань-Самара,АвтоДорСервис,Стройматериалы,16.0,360,5.6,31380,0
3,T0004,2025-03-19,Казань-Самара,ТрансЛогистик,Металл,17.4,360,7.0,48470,1
4,T0005,2025-03-30,Екатеринбург-Челябинск,РусКарго,Металл,NaN,220,3.7,21090,0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   trip_id      400 non-null    str           
 1   date         400 non-null    datetime64[us]
 2   route        400 non-null    str           
 3   carrier      400 non-null    str           
 4   cargo_type   400 non-null    str           
 5   weight_t     386 non-null    float64       
 6   distance_km  400 non-null    int64         
 7   duration_h   400 non-null    float64       
 8   cost_rub     400 non-null    int64         
 9   delay_h      400 non-null    int64         
dtypes: datetime64[us](1), float64(2), int64(3), str(4)
memory usage: 31.4 KB


In [7]:
# Статистика по числовым столбцам
df.describe().round(1)

,date,weight_t,distance_km,duration_h,cost_rub,delay_h
count,400,386.0,400.0,400.0,400.0,400.0
mean,2025-03-15 22:33:36,12.5,527.9,11.5,43922.0,1.6
min,2025-03-01 00:00:00,3.0,220.0,3.3,5200.0,0.0
25%,2025-03-08 00:00:00,7.9,420.0,7.4,23105.0,0.0
50%,2025-03-17 00:00:00,13.0,430.0,10.6,39900.0,0.0
75%,2025-03-23 00:00:00,17.3,710.0,14.8,57802.5,2.0
max,2025-03-31 00:00:00,21.7,820.0,27.8,152950.0,12.0
std,NaN,5.4,184.9,4.9,27226.6,3.2


In [8]:
# Категории: сколько рейсов по каждому типу груза
print(df["cargo_type"].value_counts())
print()

# Пропуски по всем столбцам
print("Пропуски:")
print(df.isna().sum())
print()

# Уникальные значения
print("маршрутов:", df["route"].nunique())
print("перевозчиков:", df["carrier"].nunique())
print(df["carrier"].unique())

cargo_type
Стройматериалы    86
Контейнер         79
Металл            74
Нефтепродукты     58
Лес               55
Зерно             48
Name: count, dtype: int64

Пропуски:
trip_id         0
date            0
route           0
carrier         0
cargo_type      0
weight_t       14
distance_km     0
duration_h      0
cost_rub        0
delay_h         0
dtype: int64

маршрутов: 8
перевозчиков: 5
<StringArray>
['СибПуть', 'ВолгаТранс', 'АвтоДорСервис', 'ТрансЛогистик', 'РусКарго']
Length: 5, dtype: str


### Задание 2

Ответьте на вопросы о датасете, **каждый — одной командой**. Числа запишите в ответах.

1. Сколько в таблице строк и столбцов?
2. Какой тип груза встречается чаще всего и сколько раз?
3. В каком столбце есть пропуски и сколько их?
4. Какова максимальная стоимость рейса?
5. Какая средняя задержка по всем рейсам?

In [33]:
# 1)
print("строк и столбцов:", df.shape[0], "строк и",df.shape[1], "столбцов")

# 2)
print(df["cargo_type"].value_counts().iloc[[0]].to_string())

# 3)
df_tmp = df.isna().sum()
print(df_tmp[df_tmp > 0].to_string())

# 4)
print("максимальная стоимость:", df["cost_rub"].max())

# 5)
print("средняя задержка:", df["delay_h"].mean())

строк и столбцов: 400 строк и 10 столбцов
cargo_type
Стройматериалы    86
weight_t    14
максимальная стоимость: 152950
средняя задержка: 1.6425


---
# Часть 3. Индексация: `loc` и `iloc`

**Выбор столбца:**

| Запись | Что вернёт |
|---|---|
| `df["cost_rub"]` | `Series` — один столбец |
| `df[["cost_rub"]]` | `DataFrame` из одного столбца |
| `df[["route", "cost_rub"]]` | `DataFrame` из двух столбцов |

Разница между одинарными и двойными скобками важна: от неё зависит, что вы получите на выходе
и какие методы к этому применимы.

**Выбор строк — два разных инструмента:**

| Инструмент | Работает по | Правая граница срезa |
|---|---|---|
| `df.loc[...]` | **меткам** индекса | **включается** |
| `df.iloc[...]` | **позициям** (номерам с нуля) | не включается |

Разное поведение границы — источник бесконечных ошибок. `df.loc[0:4]` вернёт 5 строк,
`df.iloc[0:4]` — четыре.

In [34]:
# Столбцы
print(type(df["cost_rub"]), "<- одинарные скобки")
print(type(df[["cost_rub"]]), "<- двойные скобки")
print()
df[["trip_id", "route", "cost_rub"]].head(3)

<class 'pandas.Series'> <- одинарные скобки
<class 'pandas.DataFrame'> <- двойные скобки



,trip_id,route,cost_rub
0,T0001,Москва-Санкт-Петербург,20390
1,T0002,Москва-Казань,78740
2,T0003,Казань-Самара,31380


In [35]:
# iloc — по позициям. Правая граница НЕ включается
print("iloc[0:4] вернул строк:", len(df.iloc[0:4]))
print("loc[0:4] вернул строк: ", len(df.loc[0:4]))
print()

# iloc: [строки, столбцы] по номерам
print(df.iloc[0:3, 0:4])
print()

# loc: [строки, столбцы] по меткам
print(df.loc[0:2, ["trip_id", "route", "cost_rub"]])

iloc[0:4] вернул строк: 4
loc[0:4] вернул строк:  5

  trip_id       date                   route        carrier
0   T0001 2025-03-24  Москва-Санкт-Петербург        СибПуть
1   T0002 2025-03-22           Москва-Казань     ВолгаТранс
2   T0003 2025-03-26           Казань-Самара  АвтоДорСервис

  trip_id                   route  cost_rub
0   T0001  Москва-Санкт-Петербург     20390
1   T0002           Москва-Казань     78740
2   T0003           Казань-Самара     31380


In [36]:
# Осмысленный индекс: сделаем номер рейса индексом — тогда loc становится удобным
df_idx = df.set_index("trip_id")
print(df_idx.loc["T0100"])
print()
print("стоимость рейса T0100:", df_idx.loc["T0100", "cost_rub"])
print()

# Вернуть индекс в обычный столбец
print(df_idx.reset_index().head(2))

date           2025-03-23 00:00:00
route                Москва-Казань
carrier                   РусКарго
cargo_type                   Зерно
weight_t                      19.0
distance_km                    820
duration_h                    19.1
cost_rub                     89080
delay_h                          0
Name: T0100, dtype: object

стоимость рейса T0100: 89080

  trip_id       date                   route     carrier cargo_type  weight_t  \
0   T0001 2025-03-24  Москва-Санкт-Петербург     СибПуть     Металл       3.7   
1   T0002 2025-03-22           Москва-Казань  ВолгаТранс      Зерно      18.2   

   distance_km  duration_h  cost_rub  delay_h  
0          710        15.5     20390        0  
1          820        13.4     78740        0  


### Задание 3

1. Выведите столбцы `trip_id`, `carrier`, `cost_rub` для **первых 7 строк**.
2. Выведите строки с позициями 10–14 (пять строк) — все столбцы.
3. Сделайте `trip_id` индексом и выведите всю информацию о рейсе `T0250`.
4. Получите расстояние рейса `T0250` одним обращением.

In [46]:
# 1)
print("====1====")
print(df.loc[0:6, ["trip_id", "carrier", "cost_rub"]])

# 2)
print("====2====")
print(df.loc[10:14])

# 3)
print("====3====")
trips = df.set_index("trip_id")
print(trips.loc["T0250"])

# 4)
print("====4====")
print("расстояние T0250:", trips.loc["T0250", ["distance_km"]])

====1====
  trip_id        carrier  cost_rub
0   T0001        СибПуть     20390
1   T0002     ВолгаТранс     78740
2   T0003  АвтоДорСервис     31380
3   T0004  ТрансЛогистик     48470
4   T0005       РусКарго     21090
5   T0006  АвтоДорСервис     23240
6   T0007        СибПуть     27160
====2====
   trip_id       date                         route        carrier cargo_type  \
10   T0011 2025-03-09              Новосибирск-Омск        СибПуть     Металл   
11   T0012 2025-03-15        Москва-Нижний Новгород  ТрансЛогистик  Контейнер   
12   T0013 2025-03-05                 Казань-Самара     ВолгаТранс     Металл   
13   T0014 2025-03-20  Санкт-Петербург-Петрозаводск     ВолгаТранс      Зерно   
14   T0015 2025-03-02              Новосибирск-Омск       РусКарго     Металл   

    weight_t  distance_km  duration_h  cost_rub  delay_h  
10      16.4          680        12.9     82400        0  
11      13.2          420        10.7     40090        2  
12      14.6          360         7.

---
# Часть 4. Фильтрация

Фильтр в pandas — это **маска**: столбец из `True`/`False`, который подставляется в скобки.
Сначала посмотрите на саму маску, это снимает половину вопросов.

**Три правила, которые нужно запомнить сразу:**

1. Условия соединяются `&` (и), `|` (или), `~` (не) — **не** `and`, `or`, `not`.
2. Каждое условие **обязательно** в круглых скобках: `(df["a"] > 1) & (df["b"] < 2)`.
3. Если результат фильтра вы собираетесь изменять — ставьте `.copy()`, иначе pandas предупредит,
   что вы правите вид на данные, а не сами данные.

Полезные методы для условий: `.isin([...])`, `.between(a, b)`, `.isna()`, `.str.contains("...")`.

In [47]:
# Маска — это просто столбец из True/False
mask = df["distance_km"] > 700
print(mask.head())
print("подходит рейсов:", mask.sum())
print()

# Подставляем маску в скобки
long_trips = df[df["distance_km"] > 700]
print("длинные рейсы:", long_trips.shape)
print()

# Два условия: скобки обязательны, соединяем через &
delayed_long = df[(df["distance_km"] > 700) & (df["delay_h"] > 0)]
print("длинные И с задержкой:", len(delayed_long))

# Или — через |
cheap_or_short = df[(df["cost_rub"] < 20000) | (df["distance_km"] < 300)]
print("дешёвые ИЛИ короткие:", len(cheap_or_short))

0     True
1     True
2    False
3    False
4    False
Name: distance_km, dtype: bool
подходит рейсов: 109

длинные рейсы: (109, 10)

длинные И с задержкой: 37
дешёвые ИЛИ короткие: 98


In [48]:
# isin — значение входит в список
fuel_metal = df[df["cargo_type"].isin(["Нефтепродукты", "Металл"])]
print("нефтепродукты и металл:", len(fuel_metal))

# between — диапазон, включая границы
mid = df[df["distance_km"].between(400, 600)]
print("расстояние 400-600 км:", len(mid))

# isna — пропуски
print("без указанного веса:", len(df[df["weight_t"].isna()]))

# str.contains — поиск по тексту
moscow = df[df["route"].str.contains("Москва")]
print("маршруты через Москву:", len(moscow))

# ~ — отрицание
not_moscow = df[~df["route"].str.contains("Москва")]
print("маршруты без Москвы:", len(not_moscow))

нефтепродукты и металл: 132
расстояние 400-600 км: 156
без указанного веса: 14
маршруты через Москву: 201
маршруты без Москвы: 199


### Задание 4

Посчитайте, сколько рейсов удовлетворяет каждому условию:

1. Задержка 4 часа и больше.
2. Перевозчик `РусКарго` **и** стоимость больше 50 000 руб.
3. Груз `Зерно` или `Лес`, расстояние меньше 500 км.
4. Вес груза не указан (пропуск).
5. Маршруты, в названии которых есть `Санкт-Петербург`, у которых не было задержки.

In [54]:
# 1)
print("задержка >= 4 ч:", (df["delay_h"] >= 4).sum())

# 2)
print("РусКарго дороже 50000:", ((df["carrier"] == "РусКарго") & (df["cost_rub"] > 50000)).sum())

# 3)
print("зерно/лес короче 500 км:", (df["cargo_type"].isin(["Зерно", "Лес"]) & (df["distance_km"] < 500)).sum())

# 4)
print("вес не указан:", df["weight_t"].isna().sum())

# 5)
print("Санкт-Петербург без задержки:", ((df["route"].str.contains("Санкт-Петербург")) & (df["delay_h"] == 0)).sum())

задержка >= 4 ч: 73
РусКарго дороже 50000: 29
зерно/лес короче 500 км: 51
вес не указан: 14
Санкт-Петербург без задержки: 76


---
# Часть 5. Изменение структуры и вычисляемые признаки

**Вычисляемый признак** — новый столбец, посчитанный из существующих. Это основная работа
аналитика: сырые данные почти никогда не содержат нужного показателя напрямую.

| Операция | Как |
|---|---|
| новый столбец | `df["new"] = df["a"] / df["b"]` |
| удалить столбцы | `df.drop(columns=["a", "b"])` |
| переименовать | `df.rename(columns={"old": "new"})` |
| сменить тип | `df["a"].astype(int)` |
| отсортировать | `df.sort_values("cost_rub", ascending=False)` |
| разбить на категории | `pd.cut(df["a"], bins=[...], labels=[...])` |

Почти все методы **возвращают новую таблицу**, а не меняют старую. Поэтому результат нужно
присваивать: `df = df.rename(...)`. Параметр `inplace=True` существует, но пользоваться им
не стоит — он мешает строить цепочки и в новых версиях pandas от него уходят.

In [55]:
# Работаем с копией, чтобы не портить исходную таблицу
trips = df.copy()

# Вычисляемые признаки
trips["cost_per_km"] = (trips["cost_rub"] / trips["distance_km"]).round(2)
trips["speed_kmh"] = (trips["distance_km"] / trips["duration_h"]).round(1)
trips["is_delayed"] = trips["delay_h"] > 0            # сразу bool

print(trips[["trip_id", "distance_km", "cost_rub", "cost_per_km", "speed_kmh", "is_delayed"]].head())
print()
print("доля рейсов с задержкой: {:.1%}".format(trips["is_delayed"].mean()))

  trip_id  distance_km  cost_rub  cost_per_km  speed_kmh  is_delayed
0   T0001          710     20390        28.72       45.8       False
1   T0002          820     78740        96.02       61.2       False
2   T0003          360     31380        87.17       64.3       False
3   T0004          360     48470       134.64       51.4        True
4   T0005          220     21090        95.86       59.5       False

доля рейсов с задержкой: 31.2%


In [56]:
# Категоризация числового столбца
trips["distance_group"] = pd.cut(
    trips["distance_km"],
    bins=[0, 300, 600, 1000],
    labels=["короткий", "средний", "длинный"],
)
print(trips["distance_group"].value_counts())
print()

# Переименование и удаление
trips = trips.rename(columns={"cost_rub": "cost"})
print("столбцы после rename:", list(trips.columns))

# Сортировка: самые дорогие километры
print()
print(trips.sort_values("cost_per_km", ascending=False)
          .head(5)[["trip_id", "route", "cargo_type", "cost_per_km"]])

distance_group
средний     208
длинный     154
короткий     38
Name: count, dtype: int64

столбцы после rename: ['trip_id', 'date', 'route', 'carrier', 'cargo_type', 'weight_t', 'distance_km', 'duration_h', 'cost', 'delay_h', 'cost_per_km', 'speed_kmh', 'is_delayed', 'distance_group']

    trip_id                         route     cargo_type  cost_per_km
297   T0298  Санкт-Петербург-Петрозаводск  Нефтепродукты       205.93
9     T0010  Санкт-Петербург-Петрозаводск  Нефтепродукты       201.65
298   T0299                Москва-Воронеж  Нефтепродукты       200.33
258   T0259  Санкт-Петербург-Петрозаводск  Нефтепродукты       189.58
362   T0363                 Москва-Казань  Нефтепродукты       186.52


### Задание 5

Создайте в таблице `trips` новые признаки:

1. `cost_per_ton` — стоимость за тонну груза (`cost` / `weight_t`), округлить до целых.
2. `delay_group` — категория задержки: `нет` (0 часов), `малая` (1–3), `крупная` (4 и больше).
   Подсказка: `pd.cut` с `bins=[-1, 0, 3, 100]`. Подумайте, почему левая граница `-1`, а не `0`.
3. `weight_filled` — вес груза, где пропуски заменены средним весом по таблице
   (метод `.fillna(...)`).

Затем выведите 10 рейсов с самой высокой стоимостью за тонну — только столбцы
`trip_id`, `route`, `weight_t`, `cost_per_ton`.

In [68]:
# 1)
trips["cost_per_ton"] = (trips["cost"] / trips["weight_t"]).round()
# 2)
trips["delay_group"] = pd.cut(trips['delay_h'], bins=[-1, 0, 3, 100], labels=["нет", "малая", "крупная"])

# 3)
trips["weight_filled"] = trips["weight_t"].fillna(trips["weight_t"].mean())

# вывод топ-10
display(trips.sort_values("cost_per_ton",ascending=False).head(10))

,trip_id,date,route,carrier,cargo_type,weight_t,distance_km,duration_h,cost,delay_h,cost_per_km,speed_kmh,is_delayed,distance_group,cost_per_ton,delay_group,weight_filled
91,T0092,2025-03-01,Москва-Казань,СибПуть,Нефтепродукты,14.9,820,17.3,128780,0,157.05,47.4,False,длинный,8643.0,нет,14.9
38,T0039,2025-03-01,Москва-Казань,ВолгаТранс,Нефтепродукты,5.0,820,17.6,42700,0,52.07,46.6,False,длинный,8540.0,нет,5.0
362,T0363,2025-03-05,Москва-Казань,РусКарго,Нефтепродукты,18.0,820,18.5,152950,1,186.52,44.3,True,длинный,8497.0,малая,18.0
7,T0008,2025-03-12,Москва-Казань,РусКарго,Нефтепродукты,3.3,820,22.2,26260,6,32.02,36.9,True,длинный,7958.0,крупная,3.3
65,T0066,2025-03-16,Москва-Казань,АвтоДорСервис,Нефтепродукты,9.2,820,15.5,73080,0,89.12,52.9,False,длинный,7943.0,нет,9.2
39,T0040,2025-03-26,Москва-Санкт-Петербург,АвтоДорСервис,Нефтепродукты,10.5,710,25.0,73630,12,103.70,28.4,True,длинный,7012.0,крупная,10.5
100,T0101,2025-03-12,Москва-Санкт-Петербург,ВолгаТранс,Нефтепродукты,6.0,710,16.7,41780,0,58.85,42.5,False,длинный,6963.0,нет,6.0
351,T0352,2025-03-19,Москва-Казань,ТрансЛогистик,Нефтепродукты,21.6,820,12.4,147060,0,179.34,66.1,False,длинный,6808.0,нет,21.6
82,T0083,2025-03-10,Москва-Казань,СибПуть,Нефтепродукты,7.1,820,14.4,48300,0,58.90,56.9,False,длинный,6803.0,нет,7.1
300,T0301,2025-03-20,Москва-Казань,ВолгаТранс,Контейнер,19.7,820,13.8,133190,0,162.43,59.4,False,длинный,6761.0,нет,19.7


---
# Часть 6. Применение функций к строкам и столбцам

Три инструмента, и важно не путать, когда какой нужен:

| Инструмент | Применяется к | Когда использовать |
|---|---|---|
| `.map(словарь)` | значениям одного столбца | подстановка по справочнику |
| `.apply(функция)` | столбцу целиком | своя логика для одного столбца |
| `.apply(функция, axis=1)` | **каждой строке** | логика, которой нужно несколько столбцов |
| `.agg([...])` | столбцам | несколько статистик сразу |

**Важное предупреждение.** `.apply(axis=1)` — самый медленный способ в pandas: он действительно
проходит строки по одной, как обычный цикл. Если то же можно записать арифметикой по столбцам
(векторизованно) — записывайте арифметикой. Ниже мы измерим разницу.

In [70]:
# map по словарю: подставляем тариф по типу груза
tariff_map = {
    "Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
    "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9,
}
trips["tariff"] = trips["cargo_type"].map(tariff_map)
print(trips[["cargo_type", "tariff"]].head())
print()

# apply к одному столбцу: своя функция
def carrier_code(name):
    # код перевозчика: первые четыре буквы заглавными
    return name[:4].upper()

trips["carrier_code"] = trips["carrier"].apply(carrier_code)
print(trips[["carrier", "carrier_code"]].drop_duplicates())

       cargo_type  tariff
0          Металл     6.8
1           Зерно     5.1
2  Стройматериалы     5.9
3          Металл     6.8
4          Металл     6.8

         carrier carrier_code
0        СибПуть         СИБП
1     ВолгаТранс         ВОЛГ
2  АвтоДорСервис         АВТО
3  ТрансЛогистик         ТРАН
4       РусКарго         РУСК


In [71]:
# apply(axis=1): функция получает всю строку и может использовать любые столбцы
def penalty(row):
    # Штраф за задержку: для нефтепродуктов 2000 руб/час, для остального 800
    if row["delay_h"] == 0:
        return 0
    rate = 2000 if row["cargo_type"] == "Нефтепродукты" else 800
    return row["delay_h"] * rate

trips["penalty"] = trips.apply(penalty, axis=1)
print(trips[["trip_id", "cargo_type", "delay_h", "penalty"]].head(8))
print()
print("всего штрафов:", trips["penalty"].sum(), "руб.")

  trip_id      cargo_type  delay_h  penalty
0   T0001          Металл        0        0
1   T0002           Зерно        0        0
2   T0003  Стройматериалы        0        0
3   T0004          Металл        1      800
4   T0005          Металл        0        0
5   T0006   Нефтепродукты        0        0
6   T0007             Лес        0        0
7   T0008   Нефтепродукты        6    12000

всего штрафов: 613200 руб.


In [72]:
# Несколько статистик сразу по нескольким столбцам
trips[["weight_t", "distance_km", "cost", "cost_per_km"]].agg(["mean", "min", "max"]).round(1)

,weight_t,distance_km,cost,cost_per_km
mean,12.5,527.9,43922.0,84.0
min,3.0,220.0,5200.0,14.2
max,21.7,820.0,152950.0,205.9


In [73]:
# Векторизация против apply: то же самое, но по-разному
import time

def bench(fn, n=20):
    start = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - start) / n * 1000

# вариант 1: apply по строкам
apply_ms = bench(lambda: trips.apply(lambda r: r["cost"] / r["distance_km"], axis=1))

# вариант 2: арифметика по столбцам
vector_ms = bench(lambda: trips["cost"] / trips["distance_km"])

print(f"apply(axis=1): {apply_ms:.2f} мс")
print(f"векторизация:  {vector_ms:.2f} мс")
print(f"разница: {apply_ms / vector_ms:.0f} раз")

apply(axis=1): 2.93 мс
векторизация:  0.06 мс
разница: 50 раз


### Задание 6

1. Через `.map()` подставьте рейтинг перевозчика в новый столбец `carrier_rating`, используя
   словарь `ratings` ниже.
2. Напишите функцию `grade(row)`, которая по строке возвращает оценку рейса:
   - `"отлично"` — задержки нет и скорость больше 55 км/ч;
   - `"нормально"` — задержка не больше 2 часов;
   - `"плохо"` — во всех остальных случаях.

   Примените её через `.apply(..., axis=1)` в столбец `grade` и выведите `value_counts()`.
3. Посчитайте через `.agg()` среднее и максимум для столбцов `cost_per_km` и `speed_kmh`.
4. Тот же расчёт, что в пункте 2, невозможно записать одной арифметической операцией — почему?
   А вот `cost_per_km` можно. Запишите словами, в чём разница между этими двумя случаями.

In [82]:
ratings = {
    "ТрансЛогистик": 4.6, "РусКарго": 4.1, "СибПуть": 3.8,
    "ВолгаТранс": 4.4, "АвтоДорСервис": 3.5,
}

# 1)
trips["carrier_rating"] = trips["carrier"].map(ratings)

# 2)
def grade(row):
    if row["delay_h"] == 0 and row["speed_kmh"]:
        return "отлично"
    elif row['delay_h'] <=2:
        return "нормально"
    return "плохо"

trips["grade"] = trips.apply(grade, axis=1)
print(trips["grade"].value_counts())

# 3)
display(trips[["cost_per_km", "speed_kmh"]].agg(["mean", "max"]))


grade
отлично      275
плохо         86
нормально     39
Name: count, dtype: int64


,cost_per_km,speed_kmh
mean,83.991675,49.06175
max,205.930000,67.70000


> **Ответ на пункт 4:**
>
> Потому что во 2 пункте - сложное условие, а при вычислении cost_per_km - используется просто деление

---
# Часть 7. Мини-проект: карточка маршрута

Руководитель просит «карточку» по маршруту **Москва-Казань**: короткую сводку, которую можно
положить на одну страницу. Соберите её из того, что изучили.

**Что должно быть в карточке:**

1. число рейсов по маршруту;
2. суммарная выручка;
3. средняя стоимость за километр;
4. средний вес груза (пропуски в расчёт не берём);
5. доля рейсов с задержкой, в процентах;
6. самый частый тип груза;
7. таблица из 5 самых дорогих рейсов маршрута: `trip_id`, `cargo_type`, `cost`, `cost_per_km`.

Выводите аккуратно, с заголовком и выравниванием — как в отчёте, а не как отладочную печать.

In [125]:
route_name = "Москва-Казань"
rt = trips[trips["route"] == route_name].copy()

print(f"КАРТОЧКА МАРШРУТА: {route_name}")
print("-" * 44)

print("Число рейсов по маршруту:", f"{(trips["route"] == "Москва-Казань").sum():>18}")
print("Суммарная выручка:", f"{trips["cost"].sum()- trips["penalty"].sum():>25}")
print("Средняя стоимость за километр:", f"{trips["cost_per_km"].mean().round(2):>13}")
print("Средний вес груза:", f"{trips["weight_t"].dropna().mean().round(2):>25}")
print("Доля рейсов с задержкой, в процентах: {:.2%}".format((trips["delay_h"] == 0).sum() / len(trips)))
print("Самый частый тип груза:", f"{df["cargo_type"].value_counts().index[0]:>20}")
print("-" * 44)
print("Таблица из 5 самых дорогих рейсов маршрута:")
display(trips.sort_values(["cost"], ascending=False)[["trip_id", "cargo_type", "cost", "cost_per_km"]].head(5))

КАРТОЧКА МАРШРУТА: Москва-Казань
--------------------------------------------
Число рейсов по маршруту:                 52
Суммарная выручка:                  16955610
Средняя стоимость за километр:         83.99
Средний вес груза:                     12.51
Доля рейсов с задержкой, в процентах: 68.75%
Самый частый тип груза:       Стройматериалы
--------------------------------------------
Таблица из 5 самых дорогих рейсов маршрута:


,trip_id,cargo_type,cost,cost_per_km
362,T0363,Нефтепродукты,152950,186.52
351,T0352,Нефтепродукты,147060,179.34
300,T0301,Контейнер,133190,162.43
91,T0092,Нефтепродукты,128780,157.05
58,T0059,Металл,120410,146.84


---
## Бонус: взгляд вперёд

Карточку вы собрали по одному маршруту. А если нужно сразу по всем восьми? Переписывать фильтр
восемь раз не придётся — для этого есть `groupby`, и это тема следующей пары. Посмотрите,
как это выглядит: одна строка вместо восьми повторов.

In [ ]:
summary = trips.groupby("route").agg(
    trips_count=("trip_id", "count"),
    revenue=("cost", "sum"),
    avg_cost_per_km=("cost_per_km", "mean"),
    delayed_share=("is_delayed", "mean"),
).round(2).sort_values("revenue", ascending=False)

summary

---
## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 6: Series и DataFrame, карточка маршрута"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: обращение к `Series` по метке и по позиции, расчёт стоимости
- [ ] Задание 2: пять ответов о датасете с числами
- [ ] Задание 3: выборки через `loc` и `iloc`, рейс `T0250` найден по индексу
- [ ] Задание 4: пять фильтров, в каждом правильно расставлены скобки и `&` / `|`
- [ ] Задание 5: три новых признака, топ-10 по стоимости за тонну
- [ ] Задание 6: `map`, `apply(axis=1)`, `agg` и письменный ответ на пункт 4
- [ ] Задание 7: карточка маршрута оформлена
- [ ] Всё запушено в репозиторий

## Что унести с пары

Главное различие дня — между **метками и позициями** (`loc` против `iloc`) и между
**векторизацией и построчной обработкой** (арифметика по столбцам против `apply(axis=1)`).
Первое определяет, получите ли вы те строки, которые имели в виду. Второе — будет ли ваш
расчёт идти секунду или двадцать минут, когда строк станет миллион.